# TUTORIAL - Implement OVHcloud AI Endpoints models routing using LiteLLM

*Take advantage of the best of **OVHcloud AI Endpoints** with intelligent routing of requests sent to LLMs based on latency!*

This guide explains how to route your requests using [LiteLLM](https://www.litellm.ai/) when working with LLM as a Service (LLMaaS) such as [OVHcloud AI Endpoints](https://www.ovhcloud.com/en/public-cloud/ai-endpoints/).

![LiteLLM](./litellm-ai-endpoints.jpg)

**Did you know that using shared "LLMaaS" and, in general, MaaS (Model as a Service) has many advantages?**

This type of solution, offers immediate access to large models without having to manage infrastructure or scalability, while significantly reducing GPU-related costs!
On the other hand, there is unpredictable load potential, but this can be easily managed using a tool that allows you to manage routing, retries, and fallbacks to the least degraded option at any given time.

## Context

As a customer of the **OVHcloud AI Endpoints** solution, you can take advantage of the best of each LLM thanks to LiteLLM's various routing, load balancing, and fallback methods.

- **AI Endpoints in a few words...**

OVHcloud AI Endpoints is a serverless API platform that lets developers easily integrate ready-to-use AI models (LLMs, vision, speech, text, image) into applications without managing infrastructure, with a strong focus on data privacy and scalability.

- **LiteLLM, what is it?**

LiteLLM is an open-source LLM gateway and unified API tool that lets developers access more than 100 LLMs (from OpenAI, Azure, Anthropic, OVHcloud, etc.) through a single, OpenAI-compatible interface, simplifying integration, cost tracking, routing, and load-balancing across providers without switching SDKs for each one.

- **Get the best of AI Endpoints with LiteLLM!**

Shared AI Endpoints offer immediate access to large models **without managing infrastructure**, with **scalability** and **shared costs**.
As for LiteLLM, the solution allows the load to be distributed across different models so as not to depend on the load of LLMs and to take advantage of the best performance. This happens on the client side by observing actual behavior and **dynamically adapting routing**, **retries**, and **fallbacks** to the least degraded option at any given moment.

## Implementation of LiteLLM routing strategies 

The best strategy depends on the use case and what you are trying to optimize:
- perceived latency
- stability
- simplicity
- compliance with rate limits

**LiteLLM Routing Strategies summary table:**

| Objective / Use Case | Recommended Routing Strategy | Why |
|----------------------|------------------------------|-----|
| General production traffic | `simple-shuffle` (default) | Stable, low overhead, evenly distributes traffic across deployments |
| Real-time chatbot / interactive UI | `latency-based-routing` | Routes to the deployment with the lowest observed latency at runtime |
| High concurrency / high throughput | `least-busy` | Avoids deployments with too many in-flight requests |
| Strict rate limits / multi-key setups | `usage-based-routing` | Prevents hitting RPM/TPM limits and reduces 429 errors |
| Cost-sensitive batch processing | `cost-based-routing` (async) | Routes requests to the cheapest available deployment |

For this tutorial, we will show how to implement two methods using **OVHcloud AI Endpoints**:
- `latency-based-routing` 
- `least-busy`

### Step 1 - Install Python dependencies

In [ ]:
!pip install litellm==1.80.10

### Step 2 - Set up your environment

- **Import Python librairies**

In [4]:
import os
import asyncio
import requests
from litellm import Router

- **Define AI Endpoints unified URL and API key**

In [7]:
OVHCLOUD_API_URL = "https://oai.endpoints.kepler.ai.cloud.ovh.net/v1"
OVHCLOUD_API_KEY = os.getenv("OVHCLOUD_AI_ENDPOINTS_API_KEY")
assert OVHCLOUD_API_KEY is not None

- **Configure model names**

> **NOTE:** *When using **LiteLLM Router**, model names must be prefixed with a provider name
(e.g. `openai/`, `anthropic/`, `azure/`, etc) even if the backend is **not OpenAI**.*
>
> LiteLLM uses the provider prefix to:
> - Select the correct **request/response schema**
> - Apply the correct **OpenAI compatible formatting**
> - Enable routing, retries, and fallbacks consistently
> 
> In the case of **OVHcloud AI Endpoints**, the API is **OpenAI compatible** but LiteLLM still needs the `ovhcloud/` prefix. Refer to this [documentation](https://docs.litellm.ai/docs/providers/ovhcloud).

In [8]:
# list the available AI Endpoints models
url = f"{OVHCLOUD_API_URL}/models"
headers = {
    "Authorization": f"Bearer {OVHCLOUD_API_KEY}"
}

resp = requests.get(url, headers=headers)
data = resp.json()

for model in data.get("data", []):
    print(model["id"])

stable-diffusion-xl-base-v10
stabilityai/stable-diffusion-xl-base-1.0
whisper-large-v3
whisper-large-v3-turbo
ppl
Mixtral-8x7B-Instruct-v0.1
Llama-3.1-8B-Instruct
Meta-Llama-3_3-70B-Instruct
Mistral-Nemo-Instruct-2407
Mistral-Small-3.2-24B-Instruct-2506
Qwen3-32B
Mistral-7B-Instruct-v0.3
Qwen3-Coder-30B-A3B-Instruct
gpt-oss-120b
gpt-oss-20b
Qwen2.5-VL-72B-Instruct
Qwen2.5-Coder-32B-Instruct
DeepSeek-R1-Distill-Llama-70B
llava-next-mistral-7b
Meta-Llama-3_1-70B-Instruct
bge-base-en-v1.5
bge-multilingual-gemma2
bge-m3


In [16]:
# define targeted models
model_list = [
    {
        "model_name": "chat",
        "litellm_params": {
            "model": "ovhcloud/Meta-Llama-3_3-70B-Instruct",    # don't forget the `ovhcloud/` prefix
            "api_base": OVHCLOUD_API_URL,
            "api_key": OVHCLOUD_API_KEY,
        },
    },
    {
        "model_name": "chat",
        "litellm_params": {
            "model": "ovhcloud/gpt-oss-120b",    # don't forget the `ovhcloud/` prefix
            "api_base": OVHCLOUD_API_URL,
            "api_key": OVHCLOUD_API_KEY,
        },
    },
]

### Step 3 - Latency-based routing for use cases with real-time requirements (e.g. chatbot)

This strategy learns from real requests and routes traffic to the currently fastest model from a list.

In [17]:
# define routing strategy - latency based routing
router_latency = Router(
    model_list=model_list,
    routing_strategy="latency-based-routing",
    enable_pre_call_checks=True,
)

In [18]:
# user message
messages = [{"role": "user", "content": "Explain gravity to a 6 years old"}]
model = "chat"

tasks = [router_latency.acompletion(model=model, messages=messages) for _ in range(2)]
await asyncio.gather(*tasks)
await asyncio.sleep(1)

In [19]:
# get response
response = await router_latency.acompletion(model=model, messages=messages, max_tokens=2048)

print("Model used:", response.model)
print("\nModel response:", response.choices[0].message.content)

Model used: gpt-oss-120b

Model response: **Hey there! Let’s talk about gravity.**  

Imagine the Earth is a giant, super‑strong hug that we can’t see. This hug is called **gravity**, and it pulls everything toward the ground—people, toys, water, even the air!

### How it works (in kid‑friendly words)

1. **Things fall down** – When you drop a ball, it doesn’t float away like a balloon. It zooms straight to the floor because Earth’s hug (gravity) is pulling it down.
2. **We stay on the ground** – If you tried to jump super high, you’d still come back down. That’s because Earth’s invisible hug is always tugging you back.
3. **Water stays in the ocean** – Gravity keeps the water in lakes, rivers, and oceans instead of spilling off the edge of the Earth (there isn’t really an “edge” anyway!).
4. **The moon goes around us** – The moon is also being tugged by Earth’s hug, so it circles around us instead of drifting away.

### A fun picture

Think of a big, soft trampoline. If you put a heav

 ### Step 4 - Least-Busy routing (Batch / Backend)

This strategy avoids overloading endpoints and operates synchronously by selecting the model with the fewest calls in progress.

In [14]:
# define routing strategy - latency based routing
router_least_busy = Router(
    model_list=model_list,
    routing_strategy="least-busy",
)

In [15]:
# get response
response = router_least_busy.completion(
    model="chat",
    messages=[{"role": "user", "content": "Explain gravity to a 6 years old"}],
    max_tokens=2048,
)

print("Model used:", response.model)
print("\nModel response:", response.choices[0].message.content)

Model used: Meta-Llama-3_3-70B-Instruct

Model response: Across the sky, they gently roam,
Soft and white, a wondrous home.
Their shadows dance, their peaks unfold,
A majestic sight, a story told.

Within their depths, the winds do play,
As sunbeams weave, in a gentle way.
Their beauty changes, with each new day,
A constant wonder, in a drifting way.


## Conclusion

Let's finally compare these two methods!

- **Latency-based** routing selects the deployment with the lowest observed runtime latency. It requires some traffic to learn and works best in async setups, making it ideal for real-time chatbots, assistants, and user-facing UIs where perceived latency (TTFT) directly impacts the user experience.

- **Least-busy** routing selects the deployment with the fewest in-flight requests. It is simple, deterministic, and best suited for backend services, batch jobs, and high-concurrency workloads where avoiding saturation matters more than response time.

In conclusion, the main difference is that least-busy method optimizes capacity usage, while latency-based optimizes user-perceived response time.

### What next?

To go further and implement this type of solution in a "production" mode, please refer to the LiteLLM [documentation](https://docs.litellm.ai/docs/routing).